#  Lab 02

**ANN & Deep Learning**

This notebook works through three real, publicly available datasets to show how **images**, **text**, and **audio** are all ultimately represented as **numbers**, and how standard Python libraries load, transform, and analyze that numerical data.



## Setup — Install & Import Libraries

Run this once. In Google Colab most of these are already installed; the `pip install` line just makes sure everything needed is present.

In [ ]:
# Install any missing packages (safe to re-run)
!pip install -q datasets opencv-python-headless pillow transformers librosa soundfile pandas matplotlib numpy


In [ ]:
# ---- Core libraries ----
import os
import io
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Task 1: Computer Vision ----
import cv2
from PIL import Image
from datasets import load_dataset

# ---- Task 2: NLP ----
from transformers import BertTokenizer

# ---- Task 3: Audio ----
import librosa
import librosa.display
from scipy.fft import fft, fftfreq
from IPython.display import Audio, display

print("All libraries imported successfully.")


---
# Task 1 — Computer Vision: Image Processing
### Dataset: Intel Image Classification

**Goal:** Understand how an image is represented as numerical data (a grid of pixel values) and how basic computer vision libraries read, transform, and write that data.

**Dataset used:** [Intel Image Classification](https://huggingface.co/datasets/sfarrukhm/intel-image-classification) — ~25,000 natural-scene photos in 6 classes: `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`. We load it directly from the Hugging Face Hub (a public mirror of the original Kaggle dataset), so no manual download or Kaggle API key is needed.

We will use **OpenCV** (`cv2`) and **PIL** (`Pillow`) — two of the most widely used Python image libraries.

### 1. Load Image (from the Intel Image Classification dataset)



In [ ]:
# Dataset link (Hugging Face Hub mirror of the Intel Image Classification dataset)
DATASET_PATH = "sfarrukhm/intel-image-classification"

# Load in streaming mode: pulls samples on demand instead of downloading everything
intel_dataset = load_dataset(DATASET_PATH, split="train", streaming=True)

# Pull a single example from the stream
sample = next(iter(intel_dataset))

pil_image = sample["image"].convert("RGB")   # PIL Image object
class_label = sample["label"]

print("Sample image size (W x H):", pil_image.size)
print("Class label (integer id):", class_label)


In [ ]:
# Save the sample locally so we can also demonstrate loading with OpenCV
image_path = "sample_intel_image.jpg"
pil_image.save(image_path)

# Load the image using OpenCV
# cv2.imread() reads an image and returns it as a NumPy array of shape (height, width, channels)
# IMPORTANT: OpenCV loads images in BGR order (Blue, Green, Red), not RGB!
img_bgr = cv2.imread(image_path)

# Convert from BGR (OpenCV default) to RGB (the order humans/matplotlib expect)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print("Image shape (height, width, channels):", img_rgb.shape)
print("Data type of pixel values:", img_rgb.dtype)
print("Min / Max pixel value:", img_rgb.min(), "/", img_rgb.max())


### 2. Display Original RGB Image

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.title(f"Original RGB Image (Intel dataset, class id: {class_label})")
plt.axis("off")
plt.show()


### 3. Separate RGB Channels

A color image is really **three stacked grayscale images** — one grid of numbers for how much Red is at each pixel, one for Green, one for Blue. `img_rgb[:, :, 0]` slices out just the Red channel (index 0), `[:, :, 1]` is Green, `[:, :, 2]` is Blue.

In [ ]:
# Slice out each channel -- each is now a 2D array (height x width) of intensity values 0-255
R = img_rgb[:, :, 0]
G = img_rgb[:, :, 1]
B = img_rgb[:, :, 2]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(R, cmap="Reds")
axes[0].set_title("Red Channel")
axes[0].axis("off")

axes[1].imshow(G, cmap="Greens")
axes[1].set_title("Green Channel")
axes[1].axis("off")

axes[2].imshow(B, cmap="Blues")
axes[2].set_title("Blue Channel")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("Each channel shape:", R.shape, "-> a single 2D grid of intensity values (0-255)")


**What does each channel represent?**

- Every pixel in a color image is stored as **three numbers**: how much Red, Green, and Blue light combine to produce that pixel's color.
- Each value normally ranges from **0 (none of that color)** to **255 (full intensity)**, stored as an 8-bit unsigned integer (`uint8`).
- The full image is therefore a 3D NumPy array of shape `(height, width, 3)` — a stack of three 2D grids.
- Mixing different amounts of R, G, and B at each pixel location reproduces the full range of colors we see (the standard **RGB color model**).

### 4. Convert to Grayscale

In [ ]:
# cv2.cvtColor with COLOR_RGB2GRAY combines the three channels into a single
# intensity value per pixel using a weighted formula (roughly 0.299R + 0.587G + 0.114B),
# reflecting the fact that the human eye is more sensitive to green light.
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

print("Grayscale image shape (height, width):", img_gray.shape)  # note: only 2 dimensions now
print("Data type:", img_gray.dtype)

plt.figure(figsize=(5, 5))
plt.imshow(img_gray, cmap="gray")
plt.title("Grayscale Image")
plt.axis("off")
plt.show()


### 5. Image Processing — Resize & Edge Detection

Two extra, commonly used operations:

1. **Resize** — change the pixel grid's dimensions (`cv2.resize`).
2. **Edge detection** — highlight boundaries where pixel intensity changes sharply (`cv2.Canny`).

In [ ]:
# 1) Resize the grayscale image to half its original size
resized_gray = cv2.resize(img_gray, (img_gray.shape[1] // 2, img_gray.shape[0] // 2))

# 2) Edge detection on the grayscale image using the Canny algorithm
edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(resized_gray, cmap="gray")
axes[0].set_title(f"Resized Grayscale {resized_gray.shape[::-1]}")
axes[0].axis("off")

axes[1].imshow(edges, cmap="gray")
axes[1].set_title("Edge Detection (Canny)")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### 6. Convert the Grayscale Image Back to RGB

In [ ]:
# A grayscale image has only one channel; to convert it back to a 3-channel RGB image
# we duplicate that single intensity value into the R, G, and B channels equally.
img_gray_to_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

print("Reconstructed 'RGB' image shape:", img_gray_to_rgb.shape)

plt.figure(figsize=(5, 5))
plt.imshow(img_gray_to_rgb)
plt.title("Grayscale Converted Back to RGB")
plt.axis("off")
plt.show()


### 7. Explanation — RGB → Grayscale → RGB

- **RGB → Grayscale** is a *lossy* transformation: three numbers per pixel (R, G, B) are collapsed into one intensity number. Color information is permanently discarded — there is no way to know the original R, G, B mix from the gray value alone.
- **Grayscale → RGB** does **not** recover the lost color. It only duplicates the single gray value into all three channels, which is why the "converted back" image still *looks* black-and-white even though it is technically stored in a 3-channel RGB array again.
- **Library / functions used:**
  - `datasets.load_dataset()` (Hugging Face `datasets`) — streamed the Intel Image Classification dataset directly from its link.
  - `cv2.imread`, `cv2.cvtColor`, `cv2.resize`, `cv2.Canny` (OpenCV).
  - `PIL.Image` — used to decode/save the dataset's image object.

### What did we learn? (Task 1)
- An image is stored as a NumPy array of pixel intensities, shape `(height, width, channels)`.
- Splitting the RGB channels shows that color is really three separate grayscale grids combined.
- Grayscale conversion is a lossy, one-way reduction of that data; converting back to RGB does not restore the discarded color.
- Public image datasets can be streamed directly by link/handle — no manual download needed.

---
# Task 2 — NLP: Text Tokenization
### Dataset: SMS Spam Collection

**Goal:** Understand how raw text is converted into tokens and numerical representations that a model like BERT can process.

**Dataset used:** [SMS Spam Collection](https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection) — 5,574 SMS messages labeled `ham` (legitimate) or `spam`. We load it directly from a public CSV mirror on GitHub.

### 1. Load the Dataset (directly from its link)

In [ ]:
# Public CSV mirror of the UCI SMS Spam Collection dataset
SMS_URL = "https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"

# The raw file is Latin-1 encoded and has some extra empty columns -- clean both up
sms_df = pd.read_csv(SMS_URL, encoding="latin-1")
sms_df = sms_df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})

print("Dataset shape:", sms_df.shape)
print("\nLabel counts:")
print(sms_df["label"].value_counts())

sms_df.head()


### 2. Input Text — Pick a Sample Message From the Dataset

In [ ]:
# Pick one spam and one ham message from the dataset to work with
sample_spam = sms_df[sms_df["label"] == "spam"]["text"].iloc[0]
sample_ham = sms_df[sms_df["label"] == "ham"]["text"].iloc[0]

# We'll use the spam message as our main working example for tokenization
sample_text = sample_spam

print("Sample SPAM message (used below):\n", sample_text)
print("\nSample HAM message (for comparison):\n", sample_ham)


### 3. Load a Pretrained BERT Tokenizer

In [ ]:
# We use the standard 'bert-base-uncased' tokenizer from Hugging Face Transformers.
# It is trained on a large English corpus and is a natural fit for English SMS text.
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)


**Why a BERT tokenizer?**
BERT uses **WordPiece** sub-word tokenization: common words become a single token, while rarer or longer words are broken into smaller known pieces (marked with `##`). This keeps the vocabulary compact while still being able to represent almost any English word, which is exactly the kind of noisy, informal text found in SMS messages.

### 4. Generate Tokens

In [ ]:
# tokenize() splits the raw string into sub-word tokens
tokens = tokenizer.tokenize(sample_text)

print("Original text:")
print(sample_text)
print("\nTokens:")
print(tokens)
print("\nNumber of tokens:", len(tokens))


### 5. Token IDs

In [ ]:
# convert_tokens_to_ids() looks each token up in the vocabulary and returns its integer id
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Token IDs:")
print(token_ids)


### 6. Token → ID Mapping

In [ ]:
mapping_df = pd.DataFrame({"Token": tokens, "Token ID": token_ids})
mapping_df


### Special Tokens

When we tokenize text for the model (using `encode` / `__call__` instead of the raw `tokenize`), BERT automatically adds special tokens:

- **`[CLS]`** — placed at the very start of every sequence; its final hidden state is often used as a summary representation of the whole sentence (e.g., for classification tasks like spam detection).
- **`[SEP]`** — placed at the end of a sequence (and between sentence pairs); marks where a sequence/segment ends.

In [ ]:
# encode() (or calling the tokenizer directly) adds [CLS] and [SEP] automatically
encoded = tokenizer(sample_text)
full_ids = encoded["input_ids"]
full_tokens = tokenizer.convert_ids_to_tokens(full_ids)

full_mapping_df = pd.DataFrame({"Token": full_tokens, "Token ID": full_ids})
print("Full sequence including special tokens:")
full_mapping_df


### 7. Decode Token IDs Back Into Text

In [ ]:
decoded_text = tokenizer.decode(full_ids)
decoded_text_clean = tokenizer.decode(full_ids, skip_special_tokens=True)

print("Original text:\n", sample_text)
print("\nDecoded text (with special tokens):\n", decoded_text)
print("\nDecoded text (special tokens removed):\n", decoded_text_clean)


**Is the decoded text identical to the original?**
Not always exactly. `bert-base-uncased` lower-cases everything and normalizes spacing/punctuation, so the decoded text may differ slightly in casing or spacing from the original (e.g., contractions or unusual spacing get standardized). The *meaning* is preserved, but the exact byte-for-byte string may not be, because tokenization is not always a perfectly reversible process for every model.

### Vocabulary Size & a Sample of the Vocabulary

In [ ]:
print("Full BERT vocabulary size:", tokenizer.vocab_size)

# Show a small slice of the vocabulary (as an id -> token dictionary sample)
vocab = tokenizer.get_vocab()
sample_vocab_items = list(vocab.items())[1000:1010]  # arbitrary slice, skipping reserved ids
print("\nSample vocabulary entries (token -> id):")
for tok, idx in sample_vocab_items:
    print(f"  {tok!r:15s} -> {idx}")


### Explanation

- **Token:** a chunk of text — a whole word or a sub-word piece — that the tokenizer treats as one unit.
- **What a token represents:** an entry in the tokenizer's fixed vocabulary; it is the smallest unit of text the model reasons about.
- **Token ID:** the integer index of that token inside the tokenizer's vocabulary — this is the actual number fed into the neural network.
- **How text becomes numbers:** raw string → split into sub-word tokens (WordPiece) → each token looked up in a vocabulary table → sequence of integer IDs.
- **Data flow demonstrated:** `Original Text → Tokens → Token IDs → Decoded Text`.

### What did we learn? (Task 2)
- Text is converted into numbers by breaking it into sub-word tokens and mapping each one to an integer ID from a fixed vocabulary.
- `[CLS]` and `[SEP]` are structural tokens the model uses to understand sequence boundaries, not part of the original message.
- Tokenization is *mostly* reversible, but normalization (lowercasing, spacing) means the decoded text can differ slightly from the original.
- Real-world, noisy text (like SMS messages) tokenizes just as reliably as clean text, because sub-word tokenization can represent almost any input.

---
# Task 3 — Audio Processing
### Dataset: ESC-50 (Environmental Sound Classification)

**Goal:** Understand how sound is represented as numerical data, and how to analyze it in both the time domain (waveform) and frequency domain (FFT / spectrum).

**Dataset used:** [ESC-50](https://github.com/karolpiczak/ESC-50) — 2,000 labeled 5-second environmental audio clips (44.1&nbsp;kHz, mono) across 50 classes (dog bark, rain, clock tick, chainsaw, etc.). Rather than downloading the entire ~600&nbsp;MB archive, we fetch the metadata CSV and just the two audio clips we need, directly by their raw GitHub links.

### 1. Load Metadata and Pick Two Sample Clips

In [ ]:
# ESC-50 metadata: one row per clip, with filename, category (class label), fold, etc.
ESC50_META_URL = "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/meta/esc50.csv"
ESC50_AUDIO_BASE_URL = "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/"

meta_df = pd.read_csv(ESC50_META_URL)
print("Number of clips in ESC-50:", len(meta_df))
print("Number of classes:", meta_df["category"].nunique())
meta_df[["filename", "category", "fold"]].head()


In [ ]:
# Pick two contrasting sound categories to compare later
category_1 = "dog"
category_2 = "rain"

filename_1 = meta_df[meta_df["category"] == category_1]["filename"].iloc[0]
filename_2 = meta_df[meta_df["category"] == category_2]["filename"].iloc[0]

print(f"Clip for '{category_1}':", filename_1)
print(f"Clip for '{category_2}':", filename_2)


### 2. Download the Audio Clips (by direct link) and Play Them

In [ ]:
def download_clip(filename):
    """Download a single ESC-50 clip from its raw GitHub URL and save it locally."""
    url = ESC50_AUDIO_BASE_URL + filename
    local_path = filename
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(url, local_path)
    return local_path

path_1 = download_clip(filename_1)
path_2 = download_clip(filename_2)

# Load each clip as a NumPy array of amplitude samples using librosa
signal_1, sr_1 = librosa.load(path_1, sr=None)   # sr=None keeps the file's native sampling rate
signal_2, sr_2 = librosa.load(path_2, sr=None)

print(f"'{category_1}' clip -> {len(signal_1)} samples at {sr_1} Hz")
print(f"'{category_2}' clip -> {len(signal_2)} samples at {sr_2} Hz")


In [ ]:
print(f"Playing clip 1: '{category_1}' ({filename_1})")
display(Audio(signal_1, rate=sr_1))

print(f"Playing clip 2: '{category_2}' ({filename_2})")
display(Audio(signal_2, rate=sr_2))


### 3. Display Basic Audio Properties

In [ ]:
def audio_properties(signal, sr, label):
    duration = len(signal) / sr
    print(f"--- {label} ---")
    print(f"Sampling rate   : {sr} Hz")
    print(f"Duration        : {duration:.2f} seconds")
    print(f"Number of samples: {len(signal)}")
    print(f"Max amplitude   : {np.max(np.abs(signal)):.4f}")
    print()

audio_properties(signal_1, sr_1, f"'{category_1}' clip")
audio_properties(signal_2, sr_2, f"'{category_2}' clip")


### 4. Plot the Time-Domain Waveform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

librosa.display.waveshow(signal_1, sr=sr_1, ax=axes[0])
axes[0].set_title(f"Waveform: '{category_1}'")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

librosa.display.waveshow(signal_2, sr=sr_2, ax=axes[1], color="orange")
axes[1].set_title(f"Waveform: '{category_2}'")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()


### 5. Frequency Analysis with FFT

In [ ]:
def compute_spectrum(signal, sr):
    n = len(signal)
    fft_values = fft(signal)
    fft_freqs = fftfreq(n, d=1 / sr)

    # FFT output is symmetric for real-valued signals; keep only the positive-frequency half
    positive_freqs = fft_freqs[: n // 2]
    positive_magnitude = np.abs(fft_values[: n // 2])
    return positive_freqs, positive_magnitude

freqs_1, mag_1 = compute_spectrum(signal_1, sr_1)
freqs_2, mag_2 = compute_spectrum(signal_2, sr_2)

dominant_freq_1 = freqs_1[np.argmax(mag_1)]
dominant_freq_2 = freqs_2[np.argmax(mag_2)]

print(f"Dominant frequency in '{category_1}' clip : {dominant_freq_1:.1f} Hz")
print(f"Dominant frequency in '{category_2}' clip : {dominant_freq_2:.1f} Hz")


### 6. Plot the Frequency Spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(freqs_1, mag_1)
axes[0].set_title(f"Frequency Spectrum: '{category_1}'")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Magnitude")
axes[0].set_xlim(0, 5000)
axes[0].grid(True)

axes[1].plot(freqs_2, mag_2, color="orange")
axes[1].set_title(f"Frequency Spectrum: '{category_2}'")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_xlim(0, 5000)
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("Notice how the two spectra look very different: environmental sounds like a dog bark or rain")
print("are made up of many overlapping frequencies (unlike a pure sine tone), so the spectrum is a")
print("broad, noisy shape rather than a single sharp spike -- this is typical of real-world audio.")


### 7. Explanation

- **Sound signal:** physically, rapid variations in air pressure over time; digitally, a long **array of numbers**, each representing the instantaneous amplitude at one moment in time.
- **Amplitude:** how large the pressure variation is at a given instant — corresponds to **loudness**.
- **Frequency:** how many wave cycles occur per second, measured in **Hertz (Hz)** — corresponds to **pitch/tone**. Real-world sounds like a dog bark are made of *many* frequencies at once, not a single tone.
- **Sampling rate:** how many amplitude measurements are taken per second when the microphone recorded the clip. ESC-50 clips are recorded at 44,100 Hz (CD quality).
- **How the library processes the sound:** `librosa.load()` reads the WAV file and decodes it into a NumPy array of amplitude samples; `IPython.display.Audio` plays that array back; `scipy.fft.fft` decomposes it into its frequency components.
- **FFT (Fast Fourier Transform):** an algorithm that converts a time-domain signal into its frequency-domain representation, revealing *how much of each frequency* is present — this is how we can compare the very different frequency "fingerprints" of a dog bark versus rain.

### What did we learn? (Task 3)
- Real environmental audio, just like a synthetic tone, is stored digitally as a sequence of amplitude samples over time.
- Different real-world sounds have very different, complex frequency spectra — unlike a single pure tone, which has one sharp peak.
- The FFT lets us mathematically compare sounds by their frequency content, not just by looking at (or listening to) the raw waveform.
- Public audio datasets can be used efficiently by downloading only the specific clips needed, straight from their hosting link.

---
# Final Comparison — Images vs. Text vs. Audio as Numerical Data

Although images, text, and audio *feel* completely different to us as humans, every modern computer system including AI models represents all three the same underlying way: **as arrays of numbers**. What differs is *how* the raw data is turned into those numbers and what the numbers mean.

| Aspect | Image (Intel Image Classification) | Text (SMS Spam Collection) | Audio (ESC-50) |
|---|---|---|---|
| **Raw form** | Grid of pixels | Sequence of characters/words | Continuous pressure wave |
| **Numerical representation** | 2D/3D array of pixel intensities (`height × width × channels`), values 0–255 | Sequence of integer token IDs from a fixed vocabulary | 1D array of amplitude samples over time |
| **"Unit" of data** | Pixel | Token (word / sub-word) | Sample |
| **What varies per unit** | Color intensity (R, G, B) | Identity of the token (which word/piece) | Amplitude at that instant |
| **Key transformation shown** | RGB → Grayscale → RGB (channel reduction/duplication) | Text → Tokens → IDs → Decoded text | Waveform → FFT → Frequency spectrum |
| **Library used** | OpenCV / PIL / Hugging Face `datasets` | Hugging Face `transformers` (BERT tokenizer) | Librosa / SciPy |
| **Structure of the numbers** | Spatial (2D grid, neighboring pixels are related) | Sequential (order of tokens matters) | Temporal (order of samples over time matters) |

**Big picture takeaway:** Whether the input is a photo, a sentence, or an environmental sound clip, the very first step of *any* computational pipeline (including modern AI/deep learning models) is converting that input into structured numerical data  pixel arrays, token ID sequences, or amplitude samples  using standard, well-established libraries and real, publicly available datasets fetched directly by link. Once everything is numbers, models can apply the same core mathematical tools (matrix operations, neural networks, Fourier analysis, etc.) across completely different data types.